# 05 ? Student-test evaluation and model export

The tuned HistGradientBoosting model is fitted on labelled training-partition rows only, then evaluated on labelled student-test rows. Rows with label -1 are unknown and excluded from fitting and metrics, but predictions are exported for every student-test candidate. Evaluator-test data is never loaded or referenced.


In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, confusion_matrix, classification_report
)

TRAIN_CSV = Path("data/processed/train_candidates.csv")
STUDENT_CSV = Path("data/processed/student_test_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
OPT = Path("experiments/results/person_b/bayesian_optimization_best.json")
OUT = Path("experiments/results/person_b")
ARTIFACTS = Path("artifacts/person_b")
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

for p in (TRAIN_CSV, STUDENT_CSV, MANIFEST, OPT):
    if not p.is_file():
        raise FileNotFoundError(p)

train_all = pd.read_csv(TRAIN_CSV)
student_all = pd.read_csv(STUDENT_CSV)
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
opt = json.loads(OPT.read_text(encoding="utf-8"))
features = manifest["feature_columns"]

train = train_all[train_all["is_match"].isin([0, 1])].copy()
student_labelled = student_all[student_all["is_match"].isin([0, 1])].copy()
if set(train["is_match"].unique()) != {0, 1}:
    raise ValueError("Training partition must contain both labelled classes")
if set(student_labelled["is_match"].unique()) != {0, 1}:
    raise ValueError("Student-test labelled partition must contain both classes")
if opt["baseline_family"] != "hist_gradient_boosting":
    raise ValueError("Unexpected tuned model family")

X_train = train[features].astype("float32")
y_train = train["is_match"].astype("int8")
X_eval = student_labelled[features].astype("float32")
y_eval = student_labelled["is_match"].astype("int8")
params = dict(opt["best_params"])
params["random_state"] = 42

model = HistGradientBoostingClassifier(**params)
start = time.perf_counter()
model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start

start = time.perf_counter()
prob_eval = model.predict_proba(X_eval)[:, 1]
pred_eval = (prob_eval >= 0.5).astype("int8")
inference_seconds = time.perf_counter() - start

print("Student-test labelled classification report")
print(classification_report(y_eval, pred_eval, zero_division=0))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_eval, pred_eval, labels=[0, 1]))

metrics = {
    "model_family": "hist_gradient_boosting",
    "accuracy": float(accuracy_score(y_eval, pred_eval)),
    "precision": float(precision_score(y_eval, pred_eval, zero_division=0)),
    "recall": float(recall_score(y_eval, pred_eval, zero_division=0)),
    "macro_f1": float(f1_score(y_eval, pred_eval, average="macro", zero_division=0)),
    "log_loss": float(log_loss(y_eval, prob_eval, labels=[0, 1])),
    "fit_seconds": fit_seconds,
    "student_test_inference_seconds": inference_seconds,
    "train_labelled_rows": int(len(train)),
    "student_test_labelled_rows": int(len(student_labelled)),
    "student_test_unknown_rows": int((student_all["is_match"] == -1).sum()),
    "student_test_trips": int(student_all["trip_id"].nunique()),
    "evaluator_test_used": False
}

# Observation-level Top-1: score all candidates, but assess only observations
# with at least one known positive candidate.
scored = student_all[["trip_id", "observation_idx", "candidate_arc_id", "is_match"]].copy()
scored["match_probability"] = model.predict_proba(student_all[features].astype("float32"))[:, 1]
scored["predicted_match"] = (scored["match_probability"] >= 0.5).astype("int8")
grouped = scored.groupby(["trip_id", "observation_idx"])
known_max = grouped["is_match"].max()
known_groups = known_max[known_max.isin([0, 1])].index
positive_groups = known_max[known_max == 1].index
top_idx = grouped["match_probability"].idxmax()
top = scored.loc[top_idx].set_index(["trip_id", "observation_idx"])
top_known = top.loc[top.index.intersection(positive_groups)]
metrics["observation_top1_accuracy"] = float((top_known["is_match"] == 1).mean()) if len(top_known) else None
metrics["observations_with_known_positive"] = int(len(positive_groups))
metrics["observations_with_known_labels"] = int(len(known_groups))
metrics["candidate_miss_rate"] = float((known_max.loc[known_groups] == 0).mean()) if len(known_groups) else None

(OUT / "student_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
scored.to_csv(OUT / "student_test_candidate_probabilities.csv", index=False)

# Save the train-only fitted model. Student-test data is never used for fitting.
joblib.dump({
    "model": model,
    "feature_columns": features,
    "target_column": "is_match",
    "group_column": "trip_id",
    "model_family": "hist_gradient_boosting",
    "trained_on_partitions": ["train"],
    "unknown_label": -1
}, ARTIFACTS / "candidate_classifier.joblib")

metadata = {
    "model_family": "hist_gradient_boosting",
    "hyperparameters": params,
    "features": features,
    "trained_on_partitions": ["train"],
    "student_test_metrics_from_train_only_fit": metrics,
    "evaluator_test_used": False
}
(ARTIFACTS / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("\nMETRICS")
print(json.dumps(metrics, indent=2))
print("\nExported train-only model:", (ARTIFACTS / "candidate_classifier.joblib").resolve())
print("Exported all student-test candidate probabilities:", (OUT / "student_test_candidate_probabilities.csv").resolve())


## Evaluation caveats

- Candidate-level metrics and observation-level Top-1 accuracy answer different questions.
- Unknown labels (-1) are excluded from supervised fitting and candidate-level evaluation.
- Probabilities are nevertheless exported for every student-test candidate, including unknown-label rows.
- Candidate miss rate is calculated only over observations with known labels.
- Student-test results are development-stage metrics, not evaluator-test performance.
- The exported model is fitted on training-partition labelled rows only.
- Evaluator-test data is not loaded, inspected, or evaluated.
